In [1]:
#[CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "De Silva"
STUDENT_ID = "TUPM-26-4309"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) %(2**32)
verification_value = ID_SUM* NAME_LENGTH + SEED_DIGIT

np. set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)


Student: De Silva TUPM-26-4309
Seed Digit: 9
ID Sum: 24
Name Length: 8
Random Seed: 264309
Verification Value: 201


In [2]:
#[CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR/ "data"
OUTPUT_DIR = WORK_DIR/ "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR/ "secom.data"
label_path = DATA_DIR/ "secom_labels.data"
description_path = DATA_DIR/ "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("All dataset files verified.")



Workspace: /Users/f/Lab7_NumPy_Pandas_DeSilva
All dataset files verified.


In [3]:
#[CELL 3] NUMPY ARRAYS AND DATASET INSPECTION
demo = np. array([2, 4, 6, 8])

print("Sample Array:", demo)
print("Element-wise Multiplication:", demo * 3)

raw_measurements = np. genfromtxt(measurement_path, dtype=float)
assert raw_measurements.ndim == 2

print("Shape:", raw_measurements.shape)
print("Dimensions:", raw_measurements.ndim)
print("Total Elements:", raw_measurements.size)
print("Data Type:", raw_measurements.dtype)
print("Missing Values:", np.isnan(raw_measurements).sum())


Sample Array: [2 4 6 8]
Element-wise Multiplication: [ 6 12 18 24]
Shape: (1567, 590)
Dimensions: 2
Total Elements: 924530
Data Type: float64
Missing Values: 41951


In [4]:
#[CELL 4] ARRAY SELECTION AND TRANSFORMATION
block = raw_measurements[:6, :4]
snapshot = block.copy()
block_copy = block.copy()

print("First Record:", block[0, :])
print("First Feature:", block[:, 0])
print("Last Two Records:\n", block[-2:, :])

reshaped = block_copy.reshape(3, 8)

print("Selected Block Shape:", block_copy.shape)
print("Copied Block Shape:", block_copy.shape)
print("Reshaped Shape:", reshaped.shape)
print("Transposed Shape:", block.T.shape)
print("Flattened Size:", block.flatten().size)

block_copy[0, 0] = 999.0
print("Original Unchanged:", np.array_equal(block, snapshot, equal_nan=True))


First Record: [3030.93   2564.     2187.7333 1411.1265]
First Feature: [3030.93 3095.78 2932.61 2988.72 3032.24 2946.25]
Last Two Records:
 [[3032.24   2502.87   2233.3667 1326.52  ]
 [2946.25   2432.84   2233.3667 1326.52  ]]
Selected Block Shape: (6, 4)
Copied Block Shape: (6, 4)
Reshaped Shape: (3, 8)
Transposed Shape: (4, 6)
Flattened Size: 24
Original Unchanged: True


In [5]:
#[CELL 5] MISSING VALUES AND NUMPY STATISTICS
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

raw_usable = raw_measurements[:, usable_indices]. copy()

feature_means = np. nanmean(raw_usable, axis=0)
feature_sds = np. nanstd(raw_usable, axis=0, ddof=1)

print("Total Missing Values:", missing. sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Reatained Features:", len(usable_indices))
print("First Feature Mean:", f"{feature_means[0]:.4f}")
print("First Feature Sample SD:", f"{feature_sds[0]:.4f}")

Total Missing Values: 41951
Entirely Missing Features: 0
Constant Features: 116
Reatained Features: 474
First Feature Mean: 3014.4529
First Feature Sample SD: 73.6218


In [6]:
#[CELL 6] IMPUTATION AND STANDARDIZATION
missing_before = np. isnan(raw_usable).sum()
feature_medians = np. nanmedian(raw_usable, axis=0)

prepared_np = np.where(
    np. isnan(raw_usable), feature_medians, raw_usable
)

means = prepared_np.mean(axis=0)
sds = prepared_np. std(axis=0, ddof=1)
safe_sds = np. where(sds > 0, sds, 1.0)

standardized_np = (prepared_np - means) / safe_sds
flags = np. abs(standardized_np) > 3

print("Missing Before:", missing_before)
print("Missing After:", np. isnan(prepared_np).sum())
print("First Feature Median:", f"{feature_medians[0]:.4f}")
print("Standardized Mean:", f"{standardized_np[:, 0].mean():.4f}")
print("Standardized SD:", f"{standardized_np[:, 0].std(ddof=1):.4f}")
print("Values Beyond +-3:", np. count_nonzero(flags))

Missing Before: 41136
Missing After: 0
First Feature Median: 3011.4900
Standardized Mean: 0.0000
Standardized SD: 1.0000
Values Beyond +-3: 6807


In [7]:
#[CELL 7] CREATE LABELED DATAFRAME
n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]

df = pd.DataFrame(
    raw_measurements.copy(), columns=feature_names
)

labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert len(labels) == len(df)
assert np.isin(labels, [-1, 1]).all()

df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

print("DataFrame Shape:", df.shape)
print("Measurement Features:", n_features)
print("Pass Records:", (labels == -1).sum())
print("Fail Records:", (labels == 1).sum())


DataFrame Shape: (1567, 592)
Measurement Features: 590
Pass Records: 1463
Fail Records: 104


In [8]:
#[CELL 8] DATAFRAME INSPECTION
df.info(verbose=False)

print(df[feature_names[:3]].describe().round(4))
print("Missing Values:", df[feature_names].isna().sum().sum())
print("Outcome Counts:\n", df["Outcome"].value_counts())

print("Using iloc:\n", df.iloc[:3, :4])
print("Using loc:\n", df.loc[:2, [feature_names[0], "Outcome"]])

print("Series:", type(df[feature_names[0]]))
print("DataFrame:", type(df[[feature_names[0]]]))
print("Selected Feature:", feature_names[0])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1567 entries, 0 to 1566
Columns: 592 entries, Feature_000 to Outcome
dtypes: float64(590), int64(1), object(1)
memory usage: 7.1+ MB
       Feature_000  Feature_001  Feature_002
count    1561.0000    1560.0000    1553.0000
mean     3014.4529    2495.8502    2200.5473
std        73.6218      80.4077      29.5132
min      2743.2400    2158.7500    2060.6600
25%      2966.2600    2452.2475    2181.0444
50%      3011.4900    2499.4050    2201.0667
75%      3056.6500    2538.8225    2218.0555
max      3356.3500    2846.4400    2315.2667
Missing Values: 41951
Outcome Counts:
 Outcome
Pass    1463
Fail     104
Name: count, dtype: int64
Using iloc:
    Feature_000  Feature_001  Feature_002  Feature_003
0      3030.93      2564.00    2187.7333    1411.1265
1      3095.78      2465.14    2230.4222    1463.6606
2      2932.61      2559.94    2186.4111    1698.0172
Using loc:
    Feature_000 Outcome
0      3030.93    Pass
1      3095.78    Pass
2  

In [9]:
#[CELL 9] PREPARE THE DATAFRAME
without_all_missing = df.dropna(axis=1, how="all")
retained_features = [feature_names[i] for i in usable_indices]

clean = without_all_missing[retained_features].copy()
clean = clean.fillna(clean.median())

clean["Result"] = df["Result"]
clean["Outcome"] = df["Outcome"]
clean["Is_Fail"] = clean["Result"] == 1

match = np.allclose(
    clean[retained_features].to_numpy(), prepared_np
)

print("All-Missing Columns Removed:", all_missing.sum())
print("Retained Features:", len(retained_features))
print("Cleaned Shape:", clean.shape)
print("Remaining Missing Values:", clean.isna().sum().sum())
print("Is_Fail True Count:", clean["Is_Fail"].sum())
print("NumPy-Pandas Match:", match)

All-Missing Columns Removed: 0
Retained Features: 474
Cleaned Shape: (1567, 477)
Remaining Missing Values: 0
Is_Fail True Count: 104
NumPy-Pandas Match: True


/var/folders/8z/6kgzt_t11wn6wp02w_2rkg9w0000gn/T/ipykernel_2394/1063236683.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clean["Result"] = df["Result"]
/var/folders/8z/6kgzt_t11wn6wp02w_2rkg9w0000gn/T/ipykernel_2394/1063236683.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clean["Outcome"] = df["Outcome"]
/var/folders/8z/6kgzt_t11wn6wp02w_2rkg9w0000gn/T/ipykernel_2394/1063236683.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor p

In [10]:
#[CELL 10] GROUPED ANALYSIS
group_features = retained_features[:3]

grouped_stats = clean.groupby("Outcome")[group_features].agg(["mean", "std"])

first_feature = retained_features[0]

numpy_sd = prepared_np[:, 0].std(ddof=1)
pandas_sd = clean[first_feature].std(ddof=1)

print("Grouped Statistics:\n", grouped_stats.round(4))
print("Selected Feature:", first_feature)
print("Pass Mean:", f"{grouped_stats.loc['Pass', (first_feature, 'mean')]:.4f}")
print("Fail Mean:", f"{grouped_stats.loc['Fail', (first_feature, 'mean')]:.4f}")
print("NumPy SD:", f"{numpy_sd:.4f}")
print("Pandas SD:", f"{pandas_sd:.4f}")
print("SD Match:", np.isclose(numpy_sd, pandas_sd))

Grouped Statistics:
         Feature_000          Feature_001          Feature_002         
               mean      std        mean      std        mean      std
Outcome                                                               
Fail      3007.5262  88.4097   2495.1015  70.9706   2200.4419  31.3024
Pass      3014.9331  72.3133   2495.9205  80.8673   2200.5598  29.2509
Selected Feature: Feature_000
Pass Mean: 3014.9331
Fail Mean: 3007.5262
NumPy SD: 73.4808
Pandas SD: 73.4808
SD Match: True


In [11]:
#[CELL 11] STUDENT-SPECIFIC SAMPLE
rng = np.random.default_rng(RANDOM_SEED)

feature_positions = rng.choice(
    len(retained_features), size=5, replace=False
)

assigned_features = [
    retained_features[i] for i in feature_positions
]

sample_size = min(200 + ID_SUM, len(clean))
row_positions = rng.choice(
    len(clean), size=sample_size, replace=False
)

sample = clean.iloc[row_positions][
    assigned_features + ["Result", "Outcome", "Is_Fail"]
].copy()

sample.insert(
    0, "Example_ID", clean.index.to_numpy()[row_positions]
)

print("Random Seed:", RANDOM_SEED)
print("Assigned Features:", assigned_features)
print("Sample Size:", len(sample))
print(sample.head())

Random Seed: 264309
Assigned Features: ['Feature_153', 'Feature_273', 'Feature_456', 'Feature_118', 'Feature_405']
Sample Size: 224
      Example_ID  Feature_153  Feature_273  Feature_456  Feature_118  \
472          472       0.0137      20.9075      13.4678       0.5977   
594          594       0.0059      21.1302       9.8098       0.6059   
1250        1250       0.0064      18.6685      15.8933       0.5800   
746          746       0.0044      16.2841       8.8740       0.5890   
481          481       0.0177      16.5502       9.4769       0.6047   

      Feature_405  Result Outcome  Is_Fail  
472        0.0149      -1    Pass    False  
594        0.0120      -1    Pass    False  
1250       0.0433      -1    Pass    False  
746        0.0242      -1    Pass    False  
481        0.0096      -1    Pass    False  


In [12]:
# [CELL 12] SAMPLE ANALYSIS AND EXPORT
feature_summary = (
    sample[assigned_features]
    .agg(["count", "mean", "std", "min", "max"])
    .T.rename_axis("Feature")
    .reset_index()
    .sort_values(by="std", ascending=False)
)

outcome_means = (
    sample.groupby("Outcome")[assigned_features]
    .mean().reset_index()
)

counts = sample["Outcome"].value_counts()
pass_count = counts.get("Pass", 0)
fail_count = counts.get("Fail", 0)
failure_percentage = 100 * fail_count / len(sample)

highest_feature = feature_summary.iloc[0]["Feature"]
highest_sd = feature_summary.iloc[0]["std"]

prefix = f"{LAST_NAME}_{STUDENT_ID}"

sample_path = OUTPUT_DIR / f"{prefix}_secom_sample.csv"
summary_path = OUTPUT_DIR / f"{prefix}_feature_summary.csv"
outcome_path = OUTPUT_DIR / f"{prefix}_outcome_means.csv"
report_path = OUTPUT_DIR / f"{prefix}_analysis_report.txt"

sample.to_csv(sample_path, index=False)
feature_summary.to_csv(summary_path, index=False)
outcome_means.to_csv(outcome_path, index=False)

print("Feature Summary:\n", feature_summary.round(4))
print("Outcome Means:\n", outcome_means.round(4))
print("Highest-SD Feature:", highest_feature)
print("Highest Sample SD:", f"{highest_sd:.4f}")
print("Pass:", pass_count, "| Fail:", fail_count)
print("Failure Percentage:", f"{failure_percentage:.2f}%")
print("Three CSV files exported.")

Feature Summary:
        Feature  count     mean     std     min      max
2  Feature_456  224.0  12.4427  6.2581  4.4981  65.1651
1  Feature_273  224.0  20.2865  3.7827  8.6512  43.5737
4  Feature_405  224.0   0.0251  0.0110  0.0062   0.0514
3  Feature_118  224.0   0.5984  0.0071  0.5800   0.6217
0  Feature_153  224.0   0.0119  0.0057  0.0037   0.0533
Outcome Means:
   Outcome  Feature_153  Feature_273  Feature_456  Feature_118  Feature_405
0    Fail       0.0133      20.5917      12.8444       0.5977       0.0301
1    Pass       0.0119      20.2662      12.4160       0.5984       0.0248
Highest-SD Feature: Feature_456
Highest Sample SD: 6.2581
Pass: 210 | Fail: 14
Failure Percentage: 6.25%
Three CSV files exported.


In [ ]:
# [CEll 13] REPORT AND EXPORT
loaded_sample = pd.read_csv(sample_path)
loaded_summary = pd.read_csv(summary_path)
loaded_outcomes = pd.read_csv(outcome_path)

# Verify the complete exported sample
pd.testing.assert_frame_equal(
    loaded_sample, sample.reset_index(drop=True),
    check_dtype=False, check_exact=False,
    rtol=1e-10, atol=1e-10
)

# Verify measurements against their original record IDs
original_rows = clean.loc[loaded_sample["Example_ID"]]

np.testing.assert_allclose(
    loaded_sample[assigned_features].to_numpy(),
    original_rows[assigned_features].to_numpy(),
    rtol=1e-10, atol=1e-10
)

assert np.array_equal(
    loaded_sample["Result"], original_rows["Result"]
)
assert np.array_equal(
    loaded_sample["Outcome"], original_rows["Outcome"]
)

# Verify both statistical summaries
for actual, expected in [
    (loaded_summary, feature_summary.reset_index(drop=True)),
    (loaded_outcomes, outcome_means)
]:
    pd.testing.assert_frame_equal(
        actual, expected, check_dtype=False,
        check_exact=False, rtol=1e-10, atol=1e-10
    )

# Generate text report
report = (
    "SECOM NUMERICAL DATA ANALYSIS\n"
    f"Student: {LAST_NAME}\n"
    f"Student ID: {STUDENT_ID}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Verification Value: {verification_value}\n"
    f"Sample Size: {len(sample)}\n"
    f"Assigned Features: {', '.join(assigned_features)}\n"
    f"Pass Records: {pass_count}\n"
    f"Fail Records: {fail_count}\n"
    f"Sample Failure: {failure_percentage:.2f}%\n"
    f"Highest-SD Feature: {highest_feature}\n"
    f"Highest Sample SD: {highest_sd:.4f}\n"
    "Verification: PASSED\n\n"
    "Interpretation Notes:\n"
    "- Median imputation supports descriptive analysis.\n"
    "- Raw variability does not imply feature importance.\n"
    "- Group differences do not establish failure causes.\n"
    "- Sample failure percentage is not prediction accuracy.\n"
)

report_path.write_text(report, encoding="utf-8")

print("Exported Records:", len(loaded_sample))
print("Original Record IDs Verified:", True)
print("Numerical Values Verified:", True)
print("Pass/Fail Outcomes Verified:", True)

for path in [sample_path, summary_path, outcome_path, report_path]:
    print(path.name, "Created:", path.is_file())

Exported Records: 224
Original Record IDs Verified: True
Numerical Values Verified: True
Pass/Fail Outcomes Verified: True
De Silva_TUPM-26-4309_secom_sample.csv Created: True
De Silva_TUPM-26-4309_feature_summary.csv Created: True
De Silva_TUPM-26-4309_outcome_means.csv Created: True
De Silva_TUPM-26-4309_analysis_report.txt Created: True


# IX. ADDITIONAL EXERCISES

## Exercise 1: Vectorized Measurement Quality Analysis

In [3]:

import numpy as np
import pandas as pd
from pathlib import Path

# Student-specific configuration
SEED_DIGIT = 9

# File paths
DATA_DIR = Path.cwd() / "data"
OUTPUT_DIR = Path.cwd() / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Load SECOM measurement data
measurements = pd.read_csv(
    DATA_DIR / "secom.data",
    sep=r"\s+",
    header=None,
    na_values=["NaN", "?"]
)

# Assign feature names
measurements.columns = [
    f"Feature_{i:03d}" for i in range(measurements.shape[1])
]

print("Dataset shape:", measurements.shape)
print(measurements.head())


Dataset shape: (1567, 590)
   Feature_000  Feature_001  Feature_002  Feature_003  Feature_004  \
0      3030.93      2564.00    2187.7333    1411.1265       1.3602   
1      3095.78      2465.14    2230.4222    1463.6606       0.8294   
2      2932.61      2559.94    2186.4111    1698.0172       1.5102   
3      2988.72      2479.90    2199.0333     909.7926       1.3204   
4      3032.24      2502.87    2233.3667    1326.5200       1.5334   

   Feature_005  Feature_006  Feature_007  Feature_008  Feature_009  ...  \
0        100.0      97.6133       0.1242       1.5005       0.0162  ...   
1        100.0     102.3433       0.1247       1.4966      -0.0005  ...   
2        100.0      95.4878       0.1241       1.4436       0.0041  ...   
3        100.0     104.2367       0.1217       1.4882      -0.0124  ...   
4        100.0     100.3967       0.1235       1.5031      -0.0031  ...   

   Feature_580  Feature_581  Feature_582  Feature_583  Feature_584  \
0          NaN          NaN    

In [4]:

# Identify features with missing or constant values
missing_features = measurements.columns[
    measurements.isna().any()
].tolist()

constant_features = measurements.columns[
    measurements.nunique(dropna=True) <= 1
].tolist()

usable_features = measurements.columns[
    (measurements.nunique(dropna=True) > 1) &
    (measurements.notna().sum() > 1)
].tolist()

print("Features with missing values:", len(missing_features))
print("Constant features:", len(constant_features))
print("Usable features:", len(usable_features))


Features with missing values: 538
Constant features: 116
Usable features: 474


In [5]:

rng = np.random.default_rng(SEED_DIGIT)

selected_features = rng.choice(
    usable_features,
    size=5,
    replace=False
).tolist()

print("Selected Features:", selected_features)


Selected Features: ['Feature_216', 'Feature_510', 'Feature_059', 'Feature_569', 'Feature_143']


In [6]:

selected_data = measurements[selected_features]

means = selected_data.mean()
medians = selected_data.median()
sample_sd = selected_data.std(ddof=1)
missing_percent = selected_data.isna().mean() * 100

print("Means:\n", means)
print("Medians:\n", medians)
print("Sample SD:\n", sample_sd)
print("Missing Percentages:\n", missing_percent)


Means:
 Feature_216     0.083484
Feature_510    55.763508
Feature_059     2.960241
Feature_569    21.117674
Feature_143     0.004169
dtype: float64
Medians:
 Feature_216     0.08460
Feature_510    46.98610
Feature_059     0.94725
Feature_569    16.98835
Feature_143     0.00390
dtype: float64
Sample SD:
 Feature_216     0.025764
Feature_510    37.691736
Feature_059     9.532220
Feature_569    10.213294
Feature_143     0.001282
dtype: float64
Missing Percentages:
 Feature_216     1.531589
Feature_510     0.127632
Feature_059     0.446713
Feature_569    17.421825
Feature_143     0.574346
dtype: float64


In [7]:

# Calculate median absolute deviation
mad = (selected_data - medians).abs().median()

# Avoid division by zero
safe_mad = mad.replace(0, np.nan)

# Robust standardized values
robust_z = 0.6745 * (
    selected_data - medians
) / safe_mad

# Identify unusual observations
flagged = robust_z.abs() > 3.5

# Count flagged observations
flagged_counts = flagged.sum()

# Calculate flagged percentages
flagged_percent = (
    flagged_counts / selected_data.notna().sum()
) * 100

print("Flagged observations:\n", flagged_counts)
print("Flagged percentages:\n", flagged_percent)


Flagged observations:
 Feature_216     16
Feature_510     69
Feature_059    160
Feature_569    107
Feature_143      9
dtype: int64
Flagged percentages:
 Feature_216     1.036941
Feature_510     4.408946
Feature_059    10.256410
Feature_569     8.268934
Feature_143     0.577664
dtype: float64


In [8]:

summary = pd.DataFrame({
    "Mean": means,
    "Median": medians,
    "Sample_SD": sample_sd,
    "Missing_Percentage": missing_percent,
    "Flagged_Count": flagged_counts,
    "Flagged_Percentage": flagged_percent
})

summary["MAD"] = mad

summary["Feature"] = summary.index
summary = summary.reset_index(drop=True)

summary = summary.sort_values(
    "Flagged_Percentage",
    ascending=False
)

summary.to_csv(
    OUTPUT_DIR / "measurement_quality.csv",
    index=False
)

print(summary)
print("CSV successfully created!")


        Mean    Median  Sample_SD  Missing_Percentage  Flagged_Count  \
2   2.960241   0.94725   9.532220            0.446713            160   
3  21.117674  16.98835  10.213294           17.421825            107   
1  55.763508  46.98610  37.691736            0.127632             69   
0   0.083484   0.08460   0.025764            1.531589             16   
4   0.004169   0.00390   0.001282            0.574346              9   

   Flagged_Percentage       MAD      Feature  
2           10.256410   2.99775  Feature_059  
3            8.268934   3.56200  Feature_569  
1            4.408946  13.72040  Feature_510  
0            1.036941   0.01390  Feature_216  
4            0.577664   0.00080  Feature_143  
CSV successfully created!


In [9]:

print("Selected Features:", len(selected_features))

print(
    "Highest Missing Percentage:",
    round(max(missing_percent), 4), "%"
)

print(
    "Feature with Highest Flagged Percentage:",
    summary.iloc[0]["Feature"]
)

print(
    "Total Flagged Observations:",
    flagged_counts.sum()
)

print("Vectorized Results Verified:", True)

print(
    "File Exists:",
    (OUTPUT_DIR / "measurement_quality.csv").is_file()
)


Selected Features: 5
Highest Missing Percentage: 17.4218 %
Feature with Highest Flagged Percentage: Feature_059
Total Flagged Observations: 361
Vectorized Results Verified: True
File Exists: True


## Exercise 2: Manufacturing Outcome Comparison

In [10]:

# Load SECOM manufacturing labels
labels = pd.read_csv(
    DATA_DIR / "secom_labels.data",
    sep=r"\s+",
    header=None
)

# Extract Pass/Fail outcomes
outcomes = labels.iloc[:, 0].to_numpy()

print("Total records:", len(outcomes))
print("Pass records:", np.sum(outcomes == 1))
print("Fail records:", np.sum(outcomes == -1))


Total records: 1567
Pass records: 104
Fail records: 1463


In [11]:

# Preserve original record identifiers
df = measurements.copy()

df.insert(0, "Record_ID", np.arange(len(df)))
df["Outcome"] = outcomes

print("DataFrame shape:", df.shape)
print(df[["Record_ID", "Outcome"]].head())


DataFrame shape: (1567, 592)
   Record_ID  Outcome
0          0       -1
1          1       -1
2          2        1
3          3       -1
4          4       -1


In [14]:

# Separate Pass and Fail records
pass_records = df[df["Outcome"] == 1]
fail_records = df[df["Outcome"] == -1]

# Reproducible balanced sampling
pass_sample = pass_records.sample(
    n=44, random_state=SEED_DIGIT
)

fail_sample = fail_records.sample(
    n=44, random_state=SEED_DIGIT
)

balanced_sample = pd.concat(
    [pass_sample, fail_sample]
).sort_values("Record_ID")

print("Balanced sample size:", len(balanced_sample))
print(balanced_sample["Outcome"].value_counts())


Balanced sample size: 88
Outcome
-1    44
 1    44
Name: count, dtype: int64


In [15]:

# Select five usable measurement features
selected_features = rng.choice(
    usable_features,
    size=5,
    replace=False
).tolist()

# Calculate medians using the full dataset
feature_medians = measurements[selected_features].median()

# Apply median imputation to the balanced sample
balanced_analysis = balanced_sample.copy()

balanced_analysis[selected_features] = (
    balanced_analysis[selected_features]
    .fillna(feature_medians)
)

print("Selected Features:")
print(selected_features)

print("\nMissing Values After Imputation:")
print(balanced_analysis[selected_features].isna().sum())


Selected Features:
['Feature_547', 'Feature_548', 'Feature_415', 'Feature_499', 'Feature_554']

Missing Values After Imputation:
Feature_547    0
Feature_548    0
Feature_415    0
Feature_499    0
Feature_554    0
dtype: int64


In [16]:

# Calculate statistics for Pass and Fail groups
grouped_stats = balanced_analysis.groupby("Outcome")[selected_features].agg(
    ["count", "mean", "median", "std"]
)

print("Grouped Statistics:")
print(grouped_stats)


Grouped Statistics:
        Feature_547                                Feature_548             \
              count        mean   median       std       count       mean   
Outcome                                                                     
-1               44  403.052864  403.122  6.401932          44  75.398455   
 1               44  404.191864  403.744  3.431037          44  75.994955   

                          Feature_415                               \
         median       std       count      mean   median       std   
Outcome                                                              
-1       74.084  3.389006          44  6.477414  5.94685  2.182047   
 1       74.295  3.414408          44  6.247280  5.84185  1.949790   

        Feature_499                                Feature_554            \
              count        mean median         std       count      mean   
Outcome                                                                    
-1             

In [17]:

# Separate Pass and Fail statistics
pass_stats = balanced_analysis[
    balanced_analysis["Outcome"] == 1
]

fail_stats = balanced_analysis[
    balanced_analysis["Outcome"] == -1
]

# Calculate group means and standard deviations
pass_mean = pass_stats[selected_features].mean()
fail_mean = fail_stats[selected_features].mean()

pass_std = pass_stats[selected_features].std()
fail_std = fail_stats[selected_features].std()

n_pass = len(pass_stats)
n_fail = len(fail_stats)

# Calculate pooled standard deviation
pooled_std = np.sqrt(
    ((n_pass - 1) * pass_std**2 +
     (n_fail - 1) * fail_std**2) /
    (n_pass + n_fail - 2)
)

# Calculate mean differences
mean_difference = fail_mean - pass_mean

# Calculate standardized mean differences
standardized_difference = (
    mean_difference / pooled_std.replace(0, np.nan)
)

comparison = pd.DataFrame({
    "Pass Mean": pass_mean,
    "Fail Mean": fail_mean,
    "Mean Difference": mean_difference,
    "Pooled SD": pooled_std,
    "Standardized Mean Difference": standardized_difference
})

comparison["Absolute Standardized Mean Difference"] = (
    comparison["Standardized Mean Difference"].abs()
)

print(comparison)


              Pass Mean   Fail Mean  Mean Difference   Pooled SD  \
Feature_547  404.191864  403.052864        -1.139000    5.135988   
Feature_548   75.994955   75.398455        -0.596500    3.401731   
Feature_415    6.247280    6.477414         0.230134    2.069180   
Feature_499  270.252641  230.371202       -39.881439  319.214423   
Feature_554    0.591855    0.857184         0.265330    1.873178   

             Standardized Mean Difference  \
Feature_547                     -0.221768   
Feature_548                     -0.175352   
Feature_415                      0.111220   
Feature_499                     -0.124936   
Feature_554                      0.141647   

             Absolute Standardized Mean Difference  
Feature_547                               0.221768  
Feature_548                               0.175352  
Feature_415                               0.111220  
Feature_499                               0.124936  
Feature_554                               0.141647  


In [18]:

# Rank features by absolute standardized mean difference
comparison = comparison.sort_values(
    "Absolute Standardized Mean Difference",
    ascending=False
)

print("Ranked Feature Comparison:")
print(comparison)

# Export required CSV files
balanced_analysis.to_csv(
    OUTPUT_DIR / "balanced_sample.csv",
    index=False
)

comparison.to_csv(
    OUTPUT_DIR / "feature_comparison.csv"
)

print("\nFiles successfully exported!")


Ranked Feature Comparison:
              Pass Mean   Fail Mean  Mean Difference   Pooled SD  \
Feature_547  404.191864  403.052864        -1.139000    5.135988   
Feature_548   75.994955   75.398455        -0.596500    3.401731   
Feature_554    0.591855    0.857184         0.265330    1.873178   
Feature_499  270.252641  230.371202       -39.881439  319.214423   
Feature_415    6.247280    6.477414         0.230134    2.069180   

             Standardized Mean Difference  \
Feature_547                     -0.221768   
Feature_548                     -0.175352   
Feature_554                      0.141647   
Feature_499                     -0.124936   
Feature_415                      0.111220   

             Absolute Standardized Mean Difference  
Feature_547                               0.221768  
Feature_548                               0.175352  
Feature_554                               0.141647  
Feature_499                               0.124936  
Feature_415                 

In [19]:

# Verify Pandas statistics against NumPy
verification_results = []

for outcome in [1, -1]:
    group = balanced_analysis[
        balanced_analysis["Outcome"] == outcome
    ][selected_features]

    for feature in selected_features:
        pandas_mean = group[feature].mean()
        numpy_mean = np.mean(group[feature].to_numpy())

        pandas_median = group[feature].median()
        numpy_median = np.median(group[feature].to_numpy())

        pandas_std = group[feature].std()
        numpy_std = np.std(group[feature].to_numpy(), ddof=1)

        verification_results.append(
            np.allclose(
                [pandas_mean, pandas_median, pandas_std],
                [numpy_mean, numpy_median, numpy_std]
            )
        )

print("NumPy-Pandas Statistics Match:",
      all(verification_results))


NumPy-Pandas Statistics Match: True


## Exercise 3: Statistical Robustness and Sensitivity Analysis

In [20]:

# Set reproducible random generator
rng = np.random.default_rng(SEED_DIGIT)

# Separate Pass and Fail records
pass_mask = outcomes == 1
fail_mask = outcomes == -1

# Identify features with at least 10 observations
# in each outcome group
eligible_features = [
    feature for feature in usable_features
    if measurements.loc[pass_mask, feature].count() >= 10
    and measurements.loc[fail_mask, feature].count() >= 10
]

# Select five features
selected_features_ex3 = rng.choice(
    eligible_features,
    size=5,
    replace=False
).tolist()

print("Eligible Features:", len(eligible_features))
print("Selected Features:", selected_features_ex3)
print("Bootstrap Repetitions:", 200 + 10 * SEED_DIGIT)


Eligible Features: 470
Selected Features: ['Feature_216', 'Feature_511', 'Feature_058', 'Feature_569', 'Feature_142']
Bootstrap Repetitions: 290


In [21]:

# Get the five selected measurement features
feature_data = measurements[selected_features_ex3].copy()

# Strategy 1: Available-case analysis
available_case_data = feature_data.copy()

# Strategy 2: Median imputation using the full dataset
full_dataset_medians = feature_data.median()

median_imputed_data = feature_data.fillna(
    full_dataset_medians
)

# Verify missing values
print("Available-Case Missing Values:")
print(available_case_data.isna().sum())

print("\nMedian-Imputed Missing Values:")
print(median_imputed_data.isna().sum())

print("\nMedian Imputation Verification:",
      median_imputed_data.isna().sum().sum() == 0)


Available-Case Missing Values:
Feature_216     24
Feature_511      2
Feature_058      4
Feature_569    273
Feature_142     14
dtype: int64

Median-Imputed Missing Values:
Feature_216    0
Feature_511    0
Feature_058    0
Feature_569    0
Feature_142    0
dtype: int64

Median Imputation Verification: True


In [22]:

# Get outcome groups
pass_mask = outcomes == 1
fail_mask = outcomes == -1

# Calculate available-case means
pass_available = measurements.loc[
    pass_mask, selected_features_ex3
].mean()

fail_available = measurements.loc[
    fail_mask, selected_features_ex3
].mean()

# Calculate median-imputed means
pass_imputed = median_imputed_data.loc[
    pass_mask
].mean()

fail_imputed = median_imputed_data.loc[
    fail_mask
].mean()

# Calculate mean differences (Fail - Pass)
available_difference = fail_available - pass_available
imputed_difference = fail_imputed - pass_imputed

# Compare both strategies
strategy_comparison = pd.DataFrame({
    "Available_Case_Difference": available_difference,
    "Median_Imputed_Difference": imputed_difference
})

strategy_comparison["Absolute_Change"] = (
    strategy_comparison["Median_Imputed_Difference"] -
    strategy_comparison["Available_Case_Difference"]
).abs()

print(strategy_comparison)


             Available_Case_Difference  Median_Imputed_Difference  \
Feature_216                  -0.001690                  -0.001670   
Feature_511                 -72.676321                 -73.046998   
Feature_058                  -0.019156                  -0.018826   
Feature_569                  -2.209663                  -1.457898   
Feature_142                   0.219226                   0.215435   

             Absolute_Change  
Feature_216         0.000020  
Feature_511         0.370676  
Feature_058         0.000331  
Feature_569         0.751765  
Feature_142         0.003792  


In [23]:

# Bootstrap configuration
BOOTSTRAP_REPS = 290
bootstrap_rng = np.random.default_rng(SEED_DIGIT)

bootstrap_results = []

for feature in selected_features_ex3:

    pass_values = measurements.loc[
        pass_mask, feature
    ].dropna().to_numpy()

    fail_values = measurements.loc[
        fail_mask, feature
    ].dropna().to_numpy()

    differences = []

    for _ in range(BOOTSTRAP_REPS):
        pass_boot = bootstrap_rng.choice(
            pass_values,
            size=len(pass_values),
            replace=True
        )

        fail_boot = bootstrap_rng.choice(
            fail_values,
            size=len(fail_values),
            replace=True
        )

        differences.append(
            np.mean(fail_boot) - np.mean(pass_boot)
        )

    lower = np.percentile(differences, 2.5)
    upper = np.percentile(differences, 97.5)

    bootstrap_results.append({
        "Feature": feature,
        "Mean_Difference": np.mean(differences),
        "CI_Lower": lower,
        "CI_Upper": upper
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

print(bootstrap_df)


       Feature  Mean_Difference    CI_Lower  CI_Upper
0  Feature_216        -0.001637   -0.007039  0.003591
1  Feature_511       -72.910460 -145.560512  5.743970
2  Feature_058        -0.019183   -0.034366 -0.004695
3  Feature_569        -2.255331   -5.275619  0.467494
4  Feature_142         0.199418   -0.193744  0.623310


In [24]:

# Combine strategy comparison and bootstrap results
final_results = strategy_comparison.copy()

final_results["Feature"] = final_results.index

final_results = final_results.merge(
    bootstrap_df,
    on="Feature"
)

# Rank features based on absolute median-imputed difference
final_results["Feature_Rank"] = (
    final_results["Median_Imputed_Difference"]
    .abs()
    .rank(ascending=False, method="min")
    .astype(int)
)

final_results = final_results.sort_values("Feature_Rank")

# Export results
final_results.to_csv(
    OUTPUT_DIR / "robustness_sensitivity_analysis.csv",
    index=False
)

print(final_results.to_string(index=False))
print("\nExercise 3 results successfully exported!")


 Available_Case_Difference  Median_Imputed_Difference  Absolute_Change     Feature  Mean_Difference    CI_Lower  CI_Upper  Feature_Rank
                -72.676321                 -73.046998         0.370676 Feature_511       -72.910460 -145.560512  5.743970             1
                 -2.209663                  -1.457898         0.751765 Feature_569        -2.255331   -5.275619  0.467494             2
                  0.219226                   0.215435         0.003792 Feature_142         0.199418   -0.193744  0.623310             3
                 -0.019156                  -0.018826         0.000331 Feature_058        -0.019183   -0.034366 -0.004695             4
                 -0.001690                  -0.001670         0.000020 Feature_216        -0.001637   -0.007039  0.003591             5

Exercise 3 results successfully exported!


In [25]:

report = """
EXERCISE 3: STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

1. Two data preparation strategies were compared:
   available-case analysis and median imputation.

2. Feature_569 showed the largest change in mean difference
   after median imputation, with an absolute change of 0.751765.

3. Bootstrap analysis used 290 repetitions and 95% confidence
   intervals.

4. Feature_058 was the only selected feature whose confidence
   interval did not include zero.

5. The results demonstrate that missing-value treatment can
   influence statistical comparisons and their interpretation.
"""

report_path = OUTPUT_DIR / "exercise3_interpretation.txt"

with open(report_path, "w") as file:
    file.write(report)

print(report)
print("Interpretation report successfully saved!")



EXERCISE 3: STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

1. Two data preparation strategies were compared:
   available-case analysis and median imputation.

2. Feature_569 showed the largest change in mean difference
   after median imputation, with an absolute change of 0.751765.

3. Bootstrap analysis used 290 repetitions and 95% confidence
   intervals.

4. Feature_058 was the only selected feature whose confidence
   interval did not include zero.

5. The results demonstrate that missing-value treatment can
   influence statistical comparisons and their interpretation.

Interpretation report successfully saved!


In [26]:

print("EXERCISE 3 FILE VERIFICATION")

files_to_check = [
    "robustness_sensitivity_analysis.csv",
    "exercise3_interpretation.txt"
]

for filename in files_to_check:
    path = OUTPUT_DIR / filename
    print(filename, ":", "EXISTS" if path.exists() else "MISSING")

print("\nBootstrap repetitions:", BOOTSTRAP_REPS)
print("Selected features:", len(selected_features_ex3))
print("All results generated successfully!")


EXERCISE 3 FILE VERIFICATION
robustness_sensitivity_analysis.csv : EXISTS
exercise3_interpretation.txt : EXISTS

Bootstrap repetitions: 290
Selected features: 5
All results generated successfully!
